<a href="https://colab.research.google.com/github/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/Multi_agent_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**🚀 Multi-Agent Systems & Workflow Patterns**

In the previous notebook [https://github.com/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/first_ai_agent.ipynb], you built a single agent that could take action. Now, you'll learn how to scale up by building agent teams.

Just like a team of people, you can create specialized agents that collaborate to solve complex problems. This is called a multi-agent system, and it's one of the most powerful concepts in AI agent development.

In this notebook, you'll:

✅ Learn when to use multi-agent systems in Agent Development Kit (ADK)

✅ Build your first system using an LLM as a "manager"

✅ Learn three core workflow patterns (Sequential, Parallel, and Loop) to coordinate your agent teams

Just like the previous notebook, you will first Setup before proceeding futher. To go through with the setup processes visit [https://github.com/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/first_ai_agent.ipynb] and see

**Section 1: Setup**

1.1: Install dependencies

1.2: Configure your Gemini API Key

1.3:  Import ADK components

1.4: Configure Retry Options

In [ ]:
pip install google-adk

In [ ]:
import os
from google.colab import userdata

try:
  GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
  os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
  print("✅ Gemini API key setup complete.")

except Exception as e:
     print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Colab secrets. Details: {e}")


✅ Gemini API key setup complete.


In [ ]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)
print("Done!")

Done!


**🤔 Section 2: Why Multi-Agent Systems? + Your First Multi-Agent**

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does research, another only writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

**2.1 Example: Research & Summarization System**
Let's build a system with two specialized agents:

Research Agent - Searches for information using Google Search

Summarizer Agent - Creates concise summaries from research findings

In [ ]:
try:
  research_agent = Agent(
    name= "ResearchAgent",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config
        ),
    instruction=""" You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.
    """,
    tools=[google_search],
    output_key= "research_findings" # Storing the result of this agent in {research_findings} key.
    )
  print("Research Agent created successfully.")
except Exception as e:
  print(f"Error creating Research Agent: {e}")



Research Agent created successfully.


In [ ]:
try:
  summary_agent = Agent(
    name= "SummaryAgent",
    model= Gemini(
         model= "gemini-2.5-flash-lite",
         retry_options= retry_config
    ),
    instruction= """Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.
    """,
    output_key="final_summary",

  )
  print("Summary Agent created Successfully!")
except Exception as e:
  print(f"Error creating Summary Agent: {e}")


Summary Agent created Successfully!


Then we bring the agents together under a root agent, or coordinator:

In [ ]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools
try:
  root_agent = Agent(
    name= "ResearchCoordinator",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config,
    ),
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.
    """,
    # We wrap the sub-agents using AgentTool so that it can be used as tool by the root_agent
    tools=[AgentTool(research_agent), AgentTool(summary_agent)]
)
  print("✅ root_agent created.")

except Exception as e:
  print(f"Error creating Root Agent/Coordinator : {e}")


✅ root_agent created.


In [ ]:
runner = InMemoryRunner(agent=root_agent)

response = await runner.run_debug("What are the latest advancements in semiconducor chip making and what do they mean for AI and why are ASML anf TSMC important for semiconductor industry?")


 ### Created new session: debug_session_id

User > What are the latest advancements in semiconducor chip making and what do they mean for AI and why are ASML anf TSMC important for semiconductor industry?


ResearchCoordinator > The latest advancements in semiconductor chip making, which include the development of smaller process nodes (like 3nm and 2nm using GAAFET technology), the integration of AI into chip design and manufacturing processes, the creation of advanced packaging solutions (such as chiplets and 3D ICs), the rise of specialized AI chips, and the exploration of emerging technologies like neuromorphic computing and silicon photonics, are significantly boosting AI capabilities. These advancements are crucial for meeting the ever-increasing computational demands of AI, especially for complex applications like generative AI and large language models.

ASML and TSMC play pivotal roles in this ecosystem. ASML is indispensable due to its virtual monopoly on the advanced Extreme Ultraviolet (EUV) lithography machines necessary for producing cutting-edge chips. TSMC, as the world's largest and most advanced semiconductor foundry, manufactures these chips at scale for leading tech co

🚥 **Section 3: Sequential Workflows - The Assembly Line**

**The Problem: Unpredictable Order**

The previous multi-agent system worked, but it relied on a detailed instruction prompt to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

**The Solution: A Fixed Pipeline**

When you need tasks to happen in a guaranteed, specific order, you can use a SequentialAgent. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

**Use Sequential when:** Order matters, you need a linear pipeline, or each step builds on the previous one.

To learn more, check out the documentation related to sequential agents in ADK.

**Architecture:** Blog Post Creation Pipeline

**3.1 Example: Blog Post Creation with Sequential Agents**

Let's build a system with three specialized agents:

**Outline Agent** - Creates a blog outline for a given topic

**Writer Agent**- Writes a blog post

**Editor Agent** - Edits a blog post draft for clarity and structure

In [ ]:
# Outline Agent: Creates the initial blog post outline
outline_agent= Agent(
    name= "OutlineAgent",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config
    ),
     instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key= "blog_outline",
)

print("✅ outline_agent created.")

✅ outline_agent created.


In [ ]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent= Agent(
    name= "WriterAgent",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config
    ),
     instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key= "blog_draft",
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [ ]:
editor_agent= Agent(
    name= "EditorAgent",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config
    ),
     instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key= "final_blog",
)

print("✅ editor_agent created.")

✅ editor_agent created.


Lets Create the root agent/coordinator that will orchestrate the whole blog pipeline.

In [ ]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


In [ ]:
runner = InMemoryRunner(agent= root_agent)

reponse = await runner.run_debug("Write a blog post about the benefits of multi-agent systems for software developers")


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of multi-agent systems for software developers
OutlineAgent > ## Outline: Unleash Your Development Superpowers: How Multi-Agent Systems Are Revolutionizing Software

### Introduction Hook:
Feeling overwhelmed by complex software projects? Are you struggling to manage intricate dependencies, scale efficiently, or adapt to ever-changing requirements? What if there was a way to build more robust, intelligent, and adaptable software by mimicking how nature solves problems? Enter Multi-Agent Systems (MAS) – a paradigm shift that's poised to transform how you develop.

### Main Section 1: Breaking Down Complexity with Decentralized Intelligence
*   **Divide and Conquer:** MAS allows you to decompose large, monolithic applications into smaller, independent, and specialized "agents." Each agent focuses on a specific task or domain, significantly reducing cognitive load for individual developers.
*   **Auto

**🛣️ Section 4: Parallel Workflows - Independent Researchers**

**The Problem: The Bottleneck**

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are not dependent on each other? For example, researching three different topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

**The Solution: Concurrent Execution**

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently.

To learn more, check out the documentation related to parallel agents in ADK.

**4.1 Example: Parallel Multi-Topic Research**

Let's build a system with four agents:

**Tech Researcher** - Researches AI/ML news and trends

**Health Researcher** - Researches recent medical news and trends

**Finance Researcher** - Researches finance and fintech news and trends

**Aggregator Agent** - Combines all research findings into a single summary

In [20]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [32]:
# Tech researcher: Researches new AL and ML trends over the internet
tech_researcher_agent= Agent(
    name= "TechResearcher",
    model= Gemini(
        model= GEMINI_MODEL,
        retry_options= retry_config,
    ),
    tools= [google_search],
    instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (50 words).
    """,
    output_key= "tech_research"
)

print("✅ Tech Research Agent created.")

✅ Tech Research Agent created.


In [33]:
# Health researcher: Researches the internet and gives breakthroughts on medical domains.
health_researcher_agent= Agent(
    name= "HealthResearcher",
    model= Gemini(
        model= GEMINI_MODEL,
        retry_options= retry_config,
    ),
    tools= [google_search],
    instruction="""Research recent medical breakthroughs. Include 3 significant advances, their practical applications, and estimated timelines. Keep the report concise (50 words).""",
    output_key= "health_research"
)

print("✅ Health Research Agent created.")

✅ Health Research Agent created.


In [34]:
# Finace researcher: Researches finance and fintech treads over the internet.
finance_researcher_agent= Agent(
    name= "FinanceResearcher",
    model= Gemini(
        model= GEMINI_MODEL,
        retry_options= retry_config,
    ),
    tools= [google_search],
    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (50 words).""",
    output_key= "finance_research"
)

print("✅ Finance Research Agent created.")

✅ Finance Research Agent created.


In [35]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results
aggregator_agent= Agent(
    name= "Aggregator_Agent",
    model= Gemini(
        model= GEMINI_MODEL,
        retry_options= retry_config,
    ),
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}

    **Health Breakthroughs:**
    {health_research}

    **Finance Innovations:**
    {finance_research}

    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 125 words.""",
    output_key= "executive_summary"
)

print("✅ Aggregator Agent created.")

✅ Aggregator Agent created.


In [36]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher_agent, health_researcher_agent, finance_researcher_agent],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


In [37]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)



 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
HealthResearcher > **Health:** AI-powered diagnostic imaging systems (like Skin Analytics' DERM for lesion analysis) offer faster, more accurate diagnoses, with potential rollouts in the UK already underway. **Tech:** Generative AI has seen mainstream adoption since late 2022, impacting coding, writing, and customer service. **Finance:** Blockchain and digital currencies are revolutionizing decentralized finance, ownership, and security, with potential applications beyond currency. Practical applications are emerging across all sectors, with timelines from immediate to within the next 5 years.
TechResearcher > **AI/ML Trends Daily Briefing**

**Key Developments:**

1.  **Generative AI & Agentic AI:** AI models are increasingly capable of creating content (text, images) and acting autonomously to complete tasks.
2.  **Ethical AI & Explainability:** Growing focus on responsib